# CREST Logging Module

CREST's logging utilities (`crest/utils/crest_logger.py` and `crest/utils/logging_service.py`) provide a single, structured logging setup for CREST itself *and* for any application that embeds it - no CREST-specific naming
convention required for your own loggers. This notebook demonstrates:

- **Your app in two rules** - a complete mini-application and the one call that sets up its logging
- **Quick start** - initializing logging once, then getting a logger anywhere
- **Scope** - the default `scope="root"` behavior vs. the opt-in `scope="crest"` isolation
- **Formats** - JSON (machine-parseable) vs. text (human-readable)
- **Per-module levels** - fine-grained verbosity control by module
- **Structured context** - attaching run/stage metadata to every log line in a block
- **Decorators** - automatically logging function calls and execution time
- **Uncaught exceptions** - capturing crashes that would otherwise be silent

See `crest/utils/crest_logger.py` and `crest/utils/logging_service.py` for the full API and docstrings; this notebook sticks to the common cases.

In [ ]:
# Allow crest to be imported
%cd -q ..

## 0. Your app in two rules

Here's a small application - a package `myapp` with one module that does work, plus an
entry point that starts the program. Setting up logging for *all* of it takes two rules:

1. **Every module** gets its own logger with one line, and does nothing else about logging:
   `log = logging.getLogger(__name__)`. No handlers, no levels, no file names.
2. **The entry point** (the one place the program starts) calls
   `LoggingService.initialize()` **once**, before doing any work.

That single call is the "central control": it decides where every module's records go
(console, JSON log file, error file), at what level, in what format, and with what
rotation - for your modules, CREST's, terrahydro's, and third-party libraries alike.
Below, the app is written to a temporary folder and run as a real program, so logger names
are real module paths (in a notebook cell they would all be `__main__`).

In [ ]:
import os, subprocess, sys, tempfile, textwrap
from pathlib import Path

app_dir = Path(tempfile.mkdtemp(prefix="crest_logger_demo_app_"))
(app_dir / "myapp").mkdir()
(app_dir / "myapp" / "__init__.py").write_text("")

# --- myapp/ingest.py: a module that does work --------------------------------
(app_dir / "myapp" / "ingest.py").write_text(textwrap.dedent("""
    import logging

    log = logging.getLogger(__name__)   # rule 1: the only logging line a module needs

    def run(n_files):
        log.info("ingesting %d files", n_files)
        log.debug("file list: %s", [f"file_{i}.nc" for i in range(n_files)])
        log.warning("1 file skipped: checksum mismatch")
"""))

# --- myapp/main.py: the entry point ------------------------------------------
(app_dir / "myapp" / "main.py").write_text(textwrap.dedent("""
    from crest.utils import LoggingService
    from myapp import ingest

    def main():
        LoggingService.initialize()     # rule 2: once, at startup, before any work
        ingest.run(3)

    if __name__ == "__main__":
        main()
"""))

def run_app(**env_overrides):
    """Run `python -m myapp.main` with extra environment variables; show console + log file."""
    log_dir = Path(tempfile.mkdtemp(prefix="crest_logger_demo_app_logs_"))
    env = {**os.environ,
           "PYTHONPATH": os.pathsep.join([str(app_dir), str(Path.cwd())]),
           "CREST_LOG_DIR": str(log_dir),
           **env_overrides}
    result = subprocess.run([sys.executable, "-m", "myapp.main"],
                            capture_output=True, text=True, env=env, cwd=app_dir)
    print("--- console (stderr) ---")
    print(result.stderr.strip())
    for f in sorted(log_dir.iterdir()):
        print(f"--- {f.name} ---")
        print(f.read_text().strip() or "(empty)")

run_app()

Every record carries its real source (`"logger": "myapp.ingest"`), the console and
`crest.log.jsonl` both receive it, and `error.jsonl` stays empty because it only collects
`ERROR` and above.

**Changing logging for the whole app without touching any module.** Because all of the
configuration lives in that one `initialize()` call, it can be changed in one place -
either by passing arguments (`log_level=`, `log_dir=`, `rotate=True`, `module_levels=`,
...) or, as below, with environment variables an operator can set at deploy time. Same
code, no edits: debug-level, human-readable text.

In [ ]:
run_app(CREST_LOG_LEVEL="DEBUG", CREST_LOG_FORMAT="text")

**Under loadEnv, the same app writes to the run file instead.** Started from a loadEnv
shell whose profile has a `logging` block, every Python process already has a run file
(`terrahydro-<pid>.log.jsonl` in `CREST_LOG_DIR`) attached by ast_utils. The same
`initialize()` call then notices it: records from every module go to that run file
**once**, CREST doesn't create `crest.log.jsonl` or `error.jsonl`, metrics go to the run
file too, and the console is unchanged. No code changes - and `runlog` shows the run.
(`run_app` above sets its own `PYTHONPATH`, so loadEnv stays off in this demo.) Details,
including the `run_log="ignore"` opt-out: the "Running under loadEnv (ast_utils)" section
of `docs/logging_guide.md`.

## 1. Quick start

`LoggingService.initialize(...)` should be called once at application startup.
Anywhere else in the code, `get_logger(__name__)` returns a logger for that module -
no special naming needed. Under the default `scope="root"`, this logger (and *any*
other logger in the process, from any package) is automatically captured.

In [ ]:
import tempfile
from pathlib import Path
from crest.utils import LoggingService, get_logger

log_dir = tempfile.mkdtemp(prefix="crest_logger_demo_")
LoggingService.initialize(log_dir=log_dir, console=True, log_level="INFO")

# In a real module this would be that module's own dotted path (e.g.
# "myapp.ingest.merra2"); every notebook cell runs as "__main__", so that's
# what shows up as the logger name below - a notebook-only quirk, not a
# CREST one.
logger = get_logger(__name__)
logger.info("Hello from the logger demo notebook")
logger.warning("Something worth a second look")

print(f"\nLog directory: {log_dir}")
print((Path(log_dir) / "crest.log.jsonl").read_text())

## 2. Scope: `scope="root"` (default) vs. `scope="crest"`

- `scope="root"` (the default) attaches CREST's handlers to the Python **root**
  logger, so *any* logger anywhere in the process - including one that has nothing
  to do with CREST and was never told about it - is captured.
- `scope="crest"` attaches handlers only to a logger literally named `"crest"`,
  isolated (`propagate=False`) from everything else. A host application's own
  loggers are *not* part of that subtree and are left alone.

`LoggingService.reset()` undoes initialization so this notebook can demonstrate both
modes cleanly, one after another - a real application only calls `initialize()` once.

In [ ]:
import logging

LoggingService.reset()
log_dir_root = tempfile.mkdtemp(prefix="crest_logger_demo_root_")
LoggingService.initialize(log_dir=log_dir_root, console=False, scope="root")

# Note: NOT named "crest" or anything CREST-related.
app_logger = logging.getLogger("myapp.ingest")
app_logger.info("my application's own log message")

content = (Path(log_dir_root) / "crest.log.jsonl").read_text()
print('scope="root" captured the app logger:', "my application's own log message" in content)

In [ ]:
LoggingService.reset()
log_dir_crest = tempfile.mkdtemp(prefix="crest_logger_demo_crest_")
LoggingService.initialize(log_dir=log_dir_crest, console=False, scope="crest")

app_logger2 = logging.getLogger("myapp.ingest")
app_logger2.info("my application's own log message, round 2")

content2 = (Path(log_dir_crest) / "crest.log.jsonl").read_text()
print('scope="crest" captured the app logger:', "round 2" in content2)
print("(False is expected here - the app's own logger is correctly isolated from CREST's handlers)")

## 3. Formats: JSON vs. text

JSON (the default) is suited to automated parsing; text is easier to read directly
during interactive development.

In [ ]:
LoggingService.reset()
log_dir_text = tempfile.mkdtemp(prefix="crest_logger_demo_text_")
LoggingService.initialize(
    log_dir=log_dir_text, console=False,
    format_type="text", log_file="crest.log.txt",
)

logging.getLogger("myapp.text_demo").info("a human-readable line")
print((Path(log_dir_text) / "crest.log.txt").read_text())

## 4. Per-module log levels

`module_levels` keys must be the real dotted module/package name of the logger being
configured (whatever that module passes to `logging.getLogger(__name__)`) - this works
the same way regardless of `scope`.

**Important nuance** (easy to assume the opposite): the app/console handlers that
`logger_setup`/`LoggingService` create have *one shared level* (`log_level`), applied
uniformly to every record that reaches them, regardless of source. `module_levels` only
controls whether a given logger *creates* a record at all - it can't make one module's
output appear at a level the shared handler would otherwise discard. So `module_levels`
is for **quieting a noisy module below the global default**, not for making one module
louder than it. Below, the global level is `DEBUG` (so everything is eligible), and
`myapp.noisy` is raised to `WARNING` to suppress its routine chatter specifically.

In [ ]:
LoggingService.reset()
log_dir_lvl = tempfile.mkdtemp(prefix="crest_logger_demo_levels_")
LoggingService.initialize(
    log_dir=log_dir_lvl, console=False, log_level="DEBUG",
    module_levels={"myapp.noisy": "WARNING"},
)

noisy  = logging.getLogger("myapp.noisy")
normal = logging.getLogger("myapp.normal")

noisy.info("noisy module: routine chatter (should NOT appear - raised to WARNING)")
noisy.warning("noisy module: something worth flagging (should appear)")
normal.debug("normal module: debug detail (should appear - overall level is DEBUG)")

print((Path(log_dir_lvl) / "crest.log.jsonl").read_text())

## 5. Structured context with `log_context`

`log_context(**kwargs)` attaches extra fields to every log record emitted inside the
`with` block (and nested blocks merge with their parent's fields). CREST's JSON
formatter has built-in slots for `stage` and `run` - use those names to have them
appear as their own top-level JSON fields; any other name still appears, just as a
plain extra field rather than through one of the formatter's named slots.

In [ ]:
from crest.utils import log_context

LoggingService.reset()
log_dir_ctx = tempfile.mkdtemp(prefix="crest_logger_demo_context_")
LoggingService.initialize(log_dir=log_dir_ctx, console=False)
logger = get_logger(__name__)

with log_context(run="exp_001", stage="training"):
    logger.info("epoch 1 complete")
    with log_context(stage="evaluation"):
        logger.info("running validation")  # run=exp_001 inherited, stage overridden

print((Path(log_dir_ctx) / "crest.log.jsonl").read_text())

## 6. Decorators: `log_function_call` and `log_execution_time`

- `@log_function_call()` logs the call signature, and the return value or raised
  exception, at the given level (default `DEBUG`).
- `@log_execution_time()` logs how long the call took (default level `INFO`), with
  the duration also available as a structured `duration_sec` field.

In [ ]:
from crest.utils import log_function_call, log_execution_time
import time

LoggingService.reset()
log_dir_dec = tempfile.mkdtemp(prefix="crest_logger_demo_decorators_")
LoggingService.initialize(log_dir=log_dir_dec, console=False, log_level="DEBUG")

@log_function_call(level="DEBUG")
def add(a, b):
    return a + b

@log_execution_time()
def slow_step():
    time.sleep(0.2)
    return "done"

add(2, 3)
slow_step()

print((Path(log_dir_dec) / "crest.log.jsonl").read_text())

## 7. Capturing uncaught exceptions

`install_global_exception_logger()` captures an exception which would otherwise only print a traceback to stderr (and potentially get lost, e.g. in a process whose stderr isn't captured). It is also logged at `CRITICAL` with its full traceback - automatically, for any exception that escapes all the way to the top of the program.

In [ ]:
import os, subprocess, sys, textwrap

log_dir_exc = tempfile.mkdtemp(prefix="crest_logger_demo_exceptions_")

script = textwrap.dedent(f'''
    from crest.utils import LoggingService, install_global_exception_logger

    LoggingService.initialize(log_dir={log_dir_exc!r}, console=False)
    install_global_exception_logger(logger_name="crest.demo")

    1 / 0  # left uncaught, exactly like a real crash would be
''')
script_path = Path(log_dir_exc) / "crash_demo.py"
script_path.write_text(script)

# PYTHONPATH=Path.cwd() so the child process can "import crest" too - a
# script run as a file (rather than "python -c") puts the *script's own*
# directory on sys.path, not the process's cwd, so cwd= alone isn't enough.
child_env = {**os.environ, "PYTHONPATH": str(Path.cwd())}
result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True, env=child_env,
)
print("child process exit code:", result.returncode)
print("\n--- child stderr (the original hook still runs too, so this is unchanged) ---")
print(result.stderr)

print("--- captured in crest.log.jsonl ---")
print((Path(log_dir_exc) / "crest.log.jsonl").read_text())

## Summary

- Call `LoggingService.initialize(...)` once at application startup; prefer it over
  calling `logger_setup()` directly, since a second `initialize()` call in the same
  process is a safe no-op unless you pass `force=True`.
- Everywhere else, just use `logging.getLogger(__name__)` (or the equivalent
  `get_logger(__name__)`) - the default `scope="root"` means it works without any
  CREST-specific naming convention.
- Reach for `scope="crest"` only if you deliberately want CREST's own logs kept
  separate from your application's.
- `module_levels`, `log_context`, `log_function_call`/`log_execution_time`, and
  `install_global_exception_logger` are optional but cheap ways to make logs more
  useful - add them where they pay for themselves, not everywhere by default.